In [2]:
import gradio as gr

In [3]:
import SAIfunc_20260217

In [4]:
import importlib
importlib.reload(SAIfunc_20260217)
Sim = SAIfunc_20260217.Sim

In [11]:
# mysimulation = Sim(nV=40, p0=1/2, pE=6/(40-1), a0=2, a1=2, nT_ini=50, nT_ext=20)
mysimulation = Sim.fromfile('data/SAIsim-20260308_1343.npz', 5)
# mysimulation = Sim(nV=10, p0=0.5, pE=0.3, a0=0.5, a1=0.5, nT_ini=5, nT_ext=3)

Read 100 from data/SAIsim-20260308_1343.npz 
 {'nV': 40, 'p0': 0.5, 'pE': 0.2, 'a0': 1.0, 'a1': 1.0}


In [17]:
def next_clicked(sliderval):
    new_val = (sliderval + 1) if (sliderval < mysimulation.len-1) else (sliderval)
    return gr.update(maximum = mysimulation.len-1, value = new_val)

def prev_clicked(sliderval):
    new_val = (sliderval - 1) if (sliderval > 0) else (sliderval)
    return gr.update(maximum = mysimulation.len-1, value = new_val)

def compute_clicked():
    mysimulation.next()
    new_max = mysimulation.len-1
    return gr.update(maximum = new_max, value = new_max)

def compute10_clicked():
    for rep in range(10):
        mysimulation.next()
    new_max = mysimulation.len-1
    return gr.update(maximum = new_max)

# def slider_changed(newsliderval, oldsliderval):
#     return mysimulation.class_diffs_json(oldsliderval, newsliderval), newsliderval

def slider_changed(newsliderval):
    return mysimulation.construct_svg(newsliderval, size_radius=300)

In [18]:
gr.close_all()

with gr.Blocks(title='SAI sim', analytics_enabled=False) as UI:
    with gr.Row():
        with gr.Column(min_width=750):
            frame = gr.HTML(mysimulation.construct_svg(0, size_radius=300))
            style_data = gr.TextArea(value='', visible='hidden')
            # displayed_tracker = gr.Number(value=0, visible='hidden')
        with gr.Column():
            with gr.Row():
                prev_btn = gr.Button('Prev', min_width=60)
                next_btn = gr.Button('Next', min_width=60)
            frame_sldr = gr.Slider(minimum=0, maximum=mysimulation.len-1, value=0, step=1, min_width=300, elem_id='SAI_slider')
            with gr.Row():
                compute_btn = gr.Button('Compute next and jump to', scale=2, min_width=250)
                compute10_btn = gr.Button('Compute 10 next', scale=1, min_width=140)

    # All EVent Handlers here:
    prev_btn.click(prev_clicked, inputs=frame_sldr, outputs=frame_sldr)
    next_btn.click(next_clicked, inputs=frame_sldr, outputs=frame_sldr)
    compute_btn.click(compute_clicked, outputs=frame_sldr)
    compute10_btn.click(compute10_clicked, outputs=frame_sldr)

    # frame_sldr.change(slider_changed, inputs=[frame_sldr, displayed_tracker], outputs=[style_data, displayed_tracker])
    frame_sldr.change(slider_changed, inputs=frame_sldr, outputs=frame)
    style_data.change(lambda anything:anything, inputs=style_data, js='(data) => update_graph(data)')

with open('update_graph.js', 'rt') as js:
    script = js.read()
    app, localurl, shareurl = UI.launch(js=script)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [22]:
gr.close_all()

Closing server running on port: 7860


In [10]:
with open('vis.svg', 'wt') as vis:
    vis.write(mysimulation.construct_svg(-10, 300))